# Interview Prep Agent

A **multi-agent system** that turns a resume PDF + a target job description into a tailored, research-backed interview prep brief.

Built with **Google ADK 2.0** (orchestration) and a custom **MCP server** (live company research).

**Pipeline:** `resume_agent` -> `research_agent` (calls the MCP server) -> `brief_agent`.

*Google x Kaggle AI Agents Intensive - Vibe Coding Capstone.*

## 1. Setup
Install dependencies and load the Gemini key from Kaggle Secrets.

> **Before running:** Internet **ON** (Settings), and add your key under **Add-ons -> Secrets** labelled `GEMINI_API_KEY`.

In [ ]:
# Setup: install dependencies and configure credentials.
# ADK reads the Gemini key from GOOGLE_API_KEY. USE_VERTEXAI=FALSE tells ADK to
# use the Google AI Studio (Gemini Developer API) backend instead of Vertex AI.
get_ipython().system('pip install -q -U google-adk google-genai mcp ddgs pypdf nest_asyncio')

import os
from kaggle_secrets import UserSecretsClient

os.environ["GOOGLE_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"

# ADK's Runner is async. nest_asyncio lets it run inside the notebook's own
# event loop (otherwise asyncio raises "loop already running").
import nest_asyncio
nest_asyncio.apply()

print("Setup complete.")

## 2. Inputs
Upload your resume PDF (**+ Add Input -> Upload**) and paste the job description.

In [ ]:
# Inputs: resume PDF + target job.
import glob

# Auto-discover an uploaded PDF (use "+ Add Input -> Upload"). Else set manually.
_pdfs = glob.glob("/kaggle/input/**/*.pdf", recursive=True) + glob.glob("/kaggle/working/**/*.pdf", recursive=True)
RESUME_PATH = _pdfs[0] if _pdfs else "REPLACE_WITH_YOUR_RESUME.pdf"
print("Resume:", RESUME_PATH)

# Change these to the interview you are actually preparing for.
TARGET_COMPANY = "Schneider Electric"
TARGET_ROLE = "Service Operations Analyst"
JOB_DESCRIPTION = "Paste the full job description text here."

## 3. The MCP server
Writes `interview_mcp_server.py` to disk. It exposes one tool, `research_company`, over the Model Context Protocol (stdio transport). The ADK agent will launch it as a subprocess.

In [ ]:
%%writefile interview_mcp_server.py
"""
Interview Prep - Company Research MCP Server
--------------------------------------------
A minimal Model Context Protocol (MCP) server exposing ONE tool, research_company,
which runs a live web search (DuckDuckGo) and returns snippets for the agent to
reason over.

Why an MCP server instead of a plain function?
  - It decouples the capability (web research) from the agent. Any MCP client
    (our ADK agent, Claude Desktop, etc.) can consume it over a standard protocol.
  - It demonstrates the client/server tool architecture the capstone asks for.

Transport: stdio. The ADK agent launches this file as a subprocess and talks to
it over standard input/output.
"""
import asyncio
import json

from mcp import types as mcp_types
from mcp.server.lowlevel import Server, NotificationOptions
from mcp.server.models import InitializationOptions
import mcp.server.stdio

from ddgs import DDGS

# Named MCP server instance.
app = Server("interview-research-mcp")


def _search_web(query, max_results=6):
    """Run a DuckDuckGo text search and return a compact, readable digest."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
    except Exception as e:  # network hiccups, rate limits, etc.
        return "[search error] " + str(e)
    if not results:
        return "[no results]"
    lines = []
    for r in results:
        lines.append("- " + r.get("title", "") + "\n  " + r.get("body", "") + "\n  (" + r.get("href", "") + ")")
    return "\n".join(lines)


@app.list_tools()
async def list_tools():
    """Advertise the tools this server exposes to any MCP client."""
    return [
        mcp_types.Tool(
            name="research_company",
            description=(
                "Research a company for a specific job interview. Returns recent "
                "web snippets about the company business, news, values, and likely "
                "interview themes for the given role."
            ),
            inputSchema={
                "type": "object",
                "properties": {
                    "company": {"type": "string", "description": "Company name"},
                    "role": {"type": "string", "description": "Target job title"},
                },
                "required": ["company", "role"],
            },
        )
    ]


@app.call_tool()
async def call_tool(name, arguments):
    """Execute a tool call requested by the MCP client (our ADK agent)."""
    if name != "research_company":
        return [mcp_types.TextContent(type="text", text=json.dumps({"error": "unknown tool " + str(name)}))]

    company = (arguments.get("company") or "").strip()
    role = (arguments.get("role") or "").strip()
    # Three focused searches beat one vague query.
    digest = "\n\n".join([
        "### " + company + " - overview\n" + _search_web(company + " company overview products"),
        "### " + company + " - recent news\n" + _search_web(company + " news 2026"),
        "### " + role + " interview\n" + _search_web(company + " " + role + " interview questions"),
    ])
    return [mcp_types.TextContent(type="text", text=digest)]


async def main():
    async with mcp.server.stdio.stdio_server() as (read_stream, write_stream):
        await app.run(
            read_stream,
            write_stream,
            InitializationOptions(
                server_name=app.name,
                server_version="0.1.0",
                capabilities=app.get_capabilities(
                    notification_options=NotificationOptions(),
                    experimental_capabilities={},
                ),
            ),
        )


if __name__ == "__main__":
    asyncio.run(main())

## 4. Resume tool
A deterministic ADK function tool: extract resume text with `pypdf`. The tool does I/O; the agent does the reasoning.

In [ ]:
# ADK tool: read the resume PDF (deterministic text extraction).
# A tool does I/O; the agent does the reasoning. Returning a {"status": ...}
# dict is the ADK-recommended shape so the model can branch on success/error.
from pypdf import PdfReader

def read_resume_text() -> dict:
    """Extract the candidate's resume text from the configured PDF file.

    Reads the notebook-level RESUME_PATH (no arguments needed) and returns the
    raw text for the agent to analyze.
    """
    try:
        reader = PdfReader(RESUME_PATH)
        text = "\n".join((page.extract_text() or "") for page in reader.pages).strip()
        if not text:
            return {"status": "error", "message": "No extractable text (is this a scanned PDF?)."}
        return {"status": "success", "resume_text": text[:12000]}
    except Exception as e:
        return {"status": "error", "message": str(e)}

## 5. Build the multi-agent pipeline
Three ADK agents chained with `SequentialAgent`. State flows between them via `output_key` + `{placeholder}` injection. The research agent consumes the MCP server; `tool_filter` restricts which tools the model can see.

In [ ]:
# Multi-agent pipeline (ADK SequentialAgent):
# resume_agent -> research_agent (via MCP server) -> brief_agent.
# Each agent writes its result to session state (output_key); the next agent
# reads it through {placeholders} injected from that state.
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

MODEL = "gemini-2.5-flash"
MCP_SERVER_PATH = os.path.abspath("interview_mcp_server.py")
USE_MCP = True  # set False to swap the MCP server for an in-process fallback tool

if USE_MCP:
    # The agent acts as an MCP client: it launches interview_mcp_server.py as a
    # subprocess and talks to it over stdio. tool_filter is a security control -
    # only the named tool is ever exposed to the model.
    research_tools = [
        McpToolset(
            connection_params=StdioConnectionParams(
                timeout=30,
                server_params=StdioServerParameters(
                    command="python3",
                    args=[MCP_SERVER_PATH],
                )
            ),
            tool_filter=["research_company"],
        )
    ]
else:
    from ddgs import DDGS
    def research_company(company: str, role: str) -> dict:
        """In-process fallback for the MCP research tool (same capability)."""
        def _s(q, n=6):
            try:
                with DDGS() as d:
                    return list(d.text(q, max_results=n))
            except Exception as e:
                return [{"title": "search error", "body": str(e)}]
        blocks = []
        for label, q in [("overview", company + " company overview products"),
                         ("news", company + " news 2026"),
                         ("interview", company + " " + role + " interview questions")]:
            items = _s(q)
            joined = "\n".join("- " + i.get("title", "") + ": " + i.get("body", "") for i in items)
            blocks.append("### " + label + "\n" + joined)
        return {"status": "success", "research": "\n\n".join(blocks)}
    research_tools = [research_company]

resume_agent = LlmAgent(
    model=MODEL,
    name="resume_agent",
    instruction=(
        "You analyze a candidate's resume. First call read_resume_text to get "
        "the resume, then output a tight structured profile: name, headline, "
        "key skills, and 4-6 achievement bullets with concrete impact. Output "
        "only that profile."
    ),
    tools=[read_resume_text],
    output_key="candidate_profile",
)

research_agent = LlmAgent(
    model=MODEL,
    name="research_agent",
    instruction=(
        "Research the company {target_company} for the role {target_role}. "
        "Call research_company with that company and role, then synthesize a "
        "concise intel brief: what they do, recent news, values, and the 4-6 "
        "competencies this interview will likely probe.\n\n"
        "Candidate profile for context:\n{candidate_profile}"
    ),
    tools=research_tools,
    output_key="company_research",
)

brief_agent = LlmAgent(
    model=MODEL,
    name="brief_agent",
    instruction=(
        "You are an elite interview coach. Using the inputs below, produce a "
        "markdown interview prep brief with: (1) a 2-line candidate snapshot, "
        "(2) company intel, (3) the 5 most likely behavioral questions, each "
        "with a tailored STAR answer grounded ONLY in the candidate's real "
        "experience, (4) 5 smart questions to ask the interviewer, (5) top 3 "
        "talking points.\n\n"
        "CANDIDATE PROFILE:\n{candidate_profile}\n\n"
        "COMPANY RESEARCH:\n{company_research}\n\n"
        "TARGET ROLE: {target_role} at {target_company}\n\n"
        "JOB DESCRIPTION:\n{job_description}"
    ),
)

root_agent = SequentialAgent(
    name="interview_prep_pipeline",
    sub_agents=[resume_agent, research_agent, brief_agent],
)
print("Pipeline built:", root_agent.name)

## 6. Run
Executes the pipeline and renders the prep brief. The trace prints which agent calls which tool - handy for the demo video.

In [ ]:
# Run the pipeline and render the brief.
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from IPython.display import Markdown, display

APP_NAME, USER_ID = "interview_prep_app", "candidate"
session_service = InMemorySessionService()

async def run_pipeline():
    # Seed session state with the inputs the agents read via {placeholders}.
    session = await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID,
        state={
            "target_company": TARGET_COMPANY,
            "target_role": TARGET_ROLE,
            "job_description": JOB_DESCRIPTION,
        },
    )
    runner = Runner(app_name=APP_NAME, agent=root_agent, session_service=session_service)
    trigger = types.Content(
        role="user",
        parts=[types.Part(text="Prepare me for the " + TARGET_ROLE + " interview at " + TARGET_COMPANY + ".")],
    )
    final_text = ""
    async for event in runner.run_async(session_id=session.id, user_id=USER_ID, new_message=trigger):
        # Trace which sub-agent calls which tool (useful for the demo video).
        try:
            if event.author and event.get_function_calls():
                for fc in event.get_function_calls():
                    print("[" + event.author + "] -> tool: " + fc.name)
        except Exception:
            pass
        if event.is_final_response() and event.content and event.content.parts:
            final_text = event.content.parts[0].text
    return final_text

brief = await run_pipeline()
with open("/kaggle/working/prep_brief.md", "w") as f:
    f.write(brief or "")
display(Markdown(brief or "_No output produced._"))
print("\nSaved to /kaggle/working/prep_brief.md")

## How this maps to the rubric

- **Agent / multi-agent system (ADK):** three coordinated `LlmAgent`s under a `SequentialAgent`.
- **MCP server:** a custom stdio MCP server (`interview_mcp_server.py`) the agent consumes as a client via `McpToolset`.
- **Tool use:** native ADK function tool (resume) + MCP tool (research), with `tool_filter` scoping.
- **Security:** no keys in code (Kaggle Secrets -> env var); MCP tool exposure restricted via `tool_filter`; resume text truncated and processed transiently.

**Extensions:** a loop agent for mock-interview follow-ups, structured-output validation, or deploying the MCP server to Cloud Run (Streamable HTTP).